[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/brent192192/batchrate-examples/blob/main/notebooks/classify-1000-rows.ipynb)

# Classify 1,000 support tickets with Batchrate

This notebook labels 1,000 support tickets as `billing`, `bug`, `how-to`, `account`, or `feature-request` using a single batch job on [Batchrate](https://batchrate.ai).

Batchrate runs OpenAI-compatible batch jobs on open models, so everything here uses the official `openai` Python package with a different `base_url`. The model is `qwen3.6-35b-a3b`, at $0.05 per 1M input tokens and $0.40 per 1M output tokens. New accounts get 10M free tokens, which covers this notebook many times over.

What you need: a Batchrate API key from the console at [batchrate.ai](https://batchrate.ai). Batches finish inside a 24-hour window and usually come back within hours, so you can submit, close the tab, and come back to collect results.

## 1. Install packages

Only the official OpenAI SDK and pandas. There is no Batchrate package.

In [ ]:
%pip install -q openai pandas

## 2. Add your API key

In Colab, open the key icon in the left sidebar, add a secret named `BATCHRATE_API_KEY`, and turn on notebook access. Outside Colab, set the `BATCHRATE_API_KEY` environment variable, or paste the key when prompted. The key is never printed or saved in the notebook.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

api_key = None
try:
    from google.colab import userdata  # only available in Colab
    api_key = userdata.get("BATCHRATE_API_KEY")
except Exception:
    pass
api_key = api_key or os.environ.get("BATCHRATE_API_KEY") or getpass("Batchrate API key: ")

client = OpenAI(api_key=api_key, base_url="https://api.batchrate.ai/v1")
MODEL = "qwen3.6-35b-a3b"
print("Client ready for", client.base_url)

## 3. Build a sample dataset

To keep this self-contained, we generate 1,000 synthetic support tickets from templates, with a fixed random seed so you get the same rows every time. Each ticket keeps the category it was generated from (`expected`), so we can check the model's labels against it at the end.

To use your own data instead, replace `df` with a DataFrame that has an `id` column and a `text` column.

In [ ]:
import random
import pandas as pd

random.seed(42)

FILL = {
    "product": ["the dashboard", "the mobile app", "the API", "the CSV export", "the Slack integration",
                "the invoice page", "the reporting tab", "the browser extension", "the scheduler", "search"],
    "plan": ["Starter", "Pro", "Team", "Business", "annual", "monthly"],
    "month": ["January", "February", "March", "April", "May", "June", "July",
              "August", "September", "October", "November", "December"],
    "amount": ["$12", "$29", "$49", "$99", "$240", "$588", "$19.99", "$350"],
    "error": ["a 500 error", "a blank white screen", "'Something went wrong'", "a spinner that never stops",
              "error code E1042", "a timeout", "a 403 Forbidden page", "duplicate rows"],
    "device": ["Chrome on Windows", "Safari on my iPhone", "Firefox", "my Android phone", "the desktop app on macOS", "Edge"],
    "task": ["export my data to Excel", "add a teammate", "set up a recurring report", "connect Google Sheets",
             "change the date format", "filter by customer", "create an API token", "set up SSO", "archive old projects"],
    "feature": ["dark mode", "a Zapier integration", "bulk editing", "custom fields", "an offline mode",
                "two-week sprints", "PDF export", "a public API for comments", "keyboard shortcuts", "multi-currency invoices"],
    "greeting": ["Hi,", "Hello,", "Hey team,", "Hi there,", "Good morning,", ""],
    "signoff": ["Thanks.", "Thanks in advance!", "Appreciate the help.", "Please advise.", "", "Cheers."],
}

TEMPLATES = {
    "billing": [
        "I was charged {amount} twice in {month}. Can you refund the duplicate charge?",
        "My invoice for {month} shows {amount} but I'm on the {plan} plan. Why the difference?",
        "I downgraded to {plan} last month but still got billed {amount}.",
        "Can I get a VAT invoice for the {amount} payment from {month}?",
        "My card was declined when renewing the {plan} plan, but the bank says it's fine. Please retry the charge.",
        "I cancelled before the trial ended and was still charged {amount}. I'd like a refund.",
        "How do I switch from monthly to annual billing and get the {plan} discount?",
    ],
    "bug": [
        "When I open {product} on {device} I get {error}.",
        "{product} has been showing {error} since this morning. Nothing changed on our side.",
        "Clicking save in {product} does nothing on {device}. No error, the change just disappears.",
        "Every export from {product} has {error} near the end of the file.",
        "{product} crashes on {device} right after login.",
        "Since the last update, {product} gives {error} whenever I filter by date.",
        "Notifications from {product} arrive twice and some links in them lead to {error}.",
    ],
    "how-to": [
        "How do I {task}?",
        "Is there a guide for how to {task}? I couldn't find it in the help center.",
        "What's the easiest way to {task} in {product}?",
        "I'm new here. Can someone walk me through how to {task}?",
        "Where is the setting to {task}? I looked all over {product}.",
        "Can I {task} from {device}, or only on desktop?",
    ],
    "account": [
        "I can't log in. The password reset email never arrives.",
        "Please change the email on my account. I left my old company and lost access to that inbox.",
        "My account got locked after too many login attempts on {device}.",
        "How do I transfer ownership of our workspace to a colleague before I leave?",
        "I'd like to delete my account and all associated data.",
        "Two-factor codes stopped working after I switched phones. I'm locked out.",
        "A former employee still has admin access. How do we remove them?",
    ],
    "feature-request": [
        "It would be great if {product} had {feature}.",
        "Any plans to add {feature}? Our team would use it daily.",
        "Feature request: {feature} in {product}.",
        "We'd upgrade to {plan} if you offered {feature}.",
        "Please consider adding {feature}. Right now we work around it with spreadsheets.",
        "Could you add {feature}? A competitor has it and our team keeps asking.",
    ],
}
WEIGHTS = {"billing": 0.22, "bug": 0.26, "how-to": 0.22, "account": 0.15, "feature-request": 0.15}
LABELS = list(TEMPLATES)

def make_ticket():
    label = random.choices(LABELS, weights=[WEIGHTS[l] for l in LABELS])[0]
    body = random.choice(TEMPLATES[label]).format(**{k: random.choice(v) for k, v in FILL.items()})
    body = body[0].upper() + body[1:]
    text = " ".join(p for p in [random.choice(FILL["greeting"]), body, random.choice(FILL["signoff"])] if p)
    return text, label

rows = [make_ticket() for _ in range(1000)]
df = pd.DataFrame({"id": [f"T{i:05d}" for i in range(1, 1001)],
                   "text": [t for t, _ in rows],
                   "expected": [l for _, l in rows]})
print(df["expected"].value_counts().to_string())
df.head(8)

## 4. Write the batch file

Each ticket becomes one line of JSONL: a normal chat completion request plus a `custom_id`, which is how results get joined back to your rows.

`response_format` with a JSON Schema and an `enum` means the model can only answer with one of the five labels, so there is no free text to clean up. `max_tokens` is kept small because the answer is tiny, and thinking is turned off explicitly with `chat_template_kwargs`.

A batch file can hold up to 5,000 lines and 10 MB, so 1,000 tickets fit in one file.

In [ ]:
import json

SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "ticket_label",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "label": {"type": "string", "enum": LABELS},
                "confidence": {"type": "string", "enum": ["low", "medium", "high"]},
            },
            "required": ["label", "confidence"],
            "additionalProperties": False,
        },
    },
}

INSTRUCTIONS = (
    "Classify the support ticket into one category. "
    "billing: charges, refunds, invoices, plans. "
    "bug: something is broken or behaves wrongly. "
    "how-to: asks how to do something that already exists. "
    "account: login, access, ownership, account settings or deletion. "
    "feature-request: asks for something the product doesn't do yet."
)

with open("tickets.jsonl", "w") as f:
    for row in df.itertuples():
        line = {
            "custom_id": row.id,  # up to 128 characters; use your own primary key
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": MODEL,
                "max_tokens": 50,
                "messages": [
                    {"role": "system", "content": INSTRUCTIONS},
                    {"role": "user", "content": row.text},
                ],
                "response_format": SCHEMA,
                "chat_template_kwargs": {"enable_thinking": False},
            },
        }
        f.write(json.dumps(line) + "\n")

size = os.path.getsize("tickets.jsonl")
n_lines = sum(1 for _ in open("tickets.jsonl"))
print(f"{n_lines} lines, {size / 1e6:.2f} MB")
print(open("tickets.jsonl").readline()[:400], "...")

## 5. Upload the file and create the batch

Two calls: upload the JSONL with `purpose="batch"`, then create a batch from it. Metadata is optional; its values must be strings.

Keep the batch id that gets printed. If your Colab runtime restarts, you can paste it into the next cell and pick up where you left off.

In [ ]:
batch_file = client.files.create(file=open("tickets.jsonl", "rb"), purpose="batch")
print("Uploaded:", batch_file.id, batch_file.status)

batch = client.batches.create(
    input_file_id=batch_file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h",
    metadata={"notebook": "classify-1000-rows", "rows": str(len(df))},
)
BATCH_ID = batch.id
print("Batch:", BATCH_ID, batch.status)

## 6. Check progress

Batches run inside a 24-hour window. Results usually come back within hours, and small batches can be held for a while so they run together with other work, so a few hours of `in_progress` is normal.

This cell checks every 60 seconds for up to 10 minutes and then stops. It's safe to re-run as often as you like, now or later. If the runtime restarted, set `BATCH_ID` to the id printed above. You can also close the notebook and come back: the batch keeps running on Batchrate either way.

In [ ]:
import time

# BATCH_ID = "batch_..."   # uncomment and paste your id if the runtime restarted
DONE = ("completed", "failed", "expired", "cancelled")

deadline = time.time() + 10 * 60
while True:
    batch = client.batches.retrieve(BATCH_ID)
    rc = batch.request_counts
    print(f"{time.strftime('%H:%M:%S')}  {batch.status:<12} "
          f"{rc.completed}/{rc.total} done, {rc.failed} failed")
    if batch.status in DONE or time.time() > deadline:
        break
    time.sleep(60)

if batch.status not in DONE:
    print("Still running. Come back later and re-run this cell.")

## 7. Download and read the results

Each output line follows the OpenAI batch result shape. The label is JSON inside `response.body.choices[0].message.content`. Each answer has a `label` (one of the five categories) and a `confidence` of `low`, `medium`, or `high`. Lines that failed are listed in a separate error file and aren't charged.

In [ ]:
assert batch.status in DONE, "The batch hasn't finished yet. Re-run the cell above later."

records = []
if batch.output_file_id:
    for line in client.files.content(batch.output_file_id).text.splitlines():
        r = json.loads(line)
        body = (r.get("response") or {}).get("body") or {}
        choice = (body.get("choices") or [{}])[0]
        content = (choice.get("message") or {}).get("content") or ""
        answer = json.loads(content) if content else {}
        usage = body.get("usage") or {}
        records.append({
            "id": r["custom_id"],
            "label": answer.get("label"),
            "confidence": answer.get("confidence"),
            "finish_reason": choice.get("finish_reason"),
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
        })

failed = {}
if batch.error_file_id:
    for line in client.files.content(batch.error_file_id).text.splitlines():
        r = json.loads(line)
        failed[r["custom_id"]] = (r.get("error") or {}).get("code")

results = pd.DataFrame(records, columns=["id", "label", "confidence", "finish_reason",
                                         "prompt_tokens", "completion_tokens"])
labeled = df.merge(results, on="id", how="left")
print(f"{len(results)} results, {len(failed)} failed")
if failed:
    print("Failure codes:", pd.Series(failed).value_counts().to_dict())
labeled["label"].value_counts(dropna=False).rename("count").to_frame()

A sample of labeled tickets, and how often the label matches the category each synthetic ticket was generated from. With real data you won't have `expected`; spot-check the `low` confidence rows instead.

In [ ]:
pd.set_option("display.max_colwidth", 90)
done = labeled.dropna(subset=["label"])
print(f"Matches expected category: {(done['label'] == done['expected']).mean():.1%} of {len(done)} labeled tickets")
labeled[["id", "text", "expected", "label", "confidence"]].sample(10, random_state=1)

## 8. What it cost

Every result line carries a `usage` object with the tokens actually used, and only completed requests are billed. Multiply by the per-token prices to get the cost of this run. New accounts spend their free tokens first, so this run most likely cost you nothing.

In [ ]:
PRICE_IN, PRICE_OUT = 0.05, 0.40  # USD per 1M tokens for qwen3.6-35b-a3b, see batchrate.ai/pricing

tok_in = int(results["prompt_tokens"].sum())
tok_out = int(results["completion_tokens"].sum())
cost = tok_in / 1e6 * PRICE_IN + tok_out / 1e6 * PRICE_OUT
n = max(len(results), 1)

print(f"Input tokens:  {tok_in:>10,}   ${tok_in / 1e6 * PRICE_IN:.4f}")
print(f"Output tokens: {tok_out:>10,}   ${tok_out / 1e6 * PRICE_OUT:.4f}")
print(f"Total:         {tok_in + tok_out:>10,}   ${cost:.4f}")
print(f"Per ticket: {(tok_in + tok_out) / n:.0f} tokens, ${cost / n:.6f}")
print(f"At this rate, 1,000,000 tickets would cost about ${cost / n * 1_000_000:,.2f}")

## 9. Next steps

- Swap in your own data: any DataFrame with `id` and `text` columns works with the cells above. Edit `LABELS` and `INSTRUCTIONS` to match your categories.
- Going past 5,000 rows: split the JSONL into several files and create one batch per file. [Classify a million rows](https://batchrate.ai/guides/classify-a-million-rows) covers splitting, account limits, and retrying failed lines by `custom_id`.
- Moving an existing OpenAI batch job: see [Run OpenAI Batch on open models](https://batchrate.ai/guides/run-openai-batch-on-open-models) and the other [guides](https://batchrate.ai/guides).
- Full API reference, including webhooks so you don't have to poll: [batchrate.ai/docs](https://batchrate.ai/docs).